# EPForecast Methodology — Training Example

This notebook demonstrates how to use the EPForecast methodology to train a multi-horizon electricity price forecasting model.

**What this notebook covers:**
1. Creating a synthetic hourly price dataset (no OMIE/REE access needed)
2. Training a `DirectMultiHorizonTrainer` on the DA1/DA2 day-ahead horizon groups
3. Inspecting cross-validation metrics
4. Running inference to generate a 7-day forecast

**To use with real data:** Replace the synthetic `ree_df` in Step 2 with your own DataFrame containing `day_ahead_price` and an hourly UTC DatetimeIndex. Optionally add weather and commodity columns for better accuracy.

---

## Setup

Install requirements from the repo root:
```bash
pip install -r requirements.txt
```

In [ ]:
import sys
import pathlib

# Add repo root to path so src/ imports work
repo_root = pathlib.Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Repo root: {repo_root}")

## Step 1 — Configure model artifact directory

Trained models are saved as `.joblib` files. Set `MODELS_DIR` to a local path.

In [ ]:
import src.models.direct_trainer as dt_module

MODELS_DIR = pathlib.Path("./example_models")
MODELS_DIR.mkdir(exist_ok=True)

# Patch the module-level constant before importing the trainer
dt_module.MODELS_DIR = MODELS_DIR
print(f"Models will be saved to: {MODELS_DIR.resolve()}")

## Step 2 — Create synthetic electricity price data

We simulate 2 years of hourly Spanish electricity prices with realistic seasonality:
- **Daily pattern:** morning peak (8-10h), evening peak (19-21h), overnight trough
- **Weekly pattern:** weekdays ~20% higher than weekends
- **Annual pattern:** winter higher (heating demand), summer moderate, spring/autumn lowest
- **Commodity trend:** slow upward drift with occasional shock events

The minimum required column is `day_ahead_price`. All other columns are optional features.

In [ ]:
def make_synthetic_prices(n_days=730, seed=42):
    """Generate synthetic hourly electricity price series."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2024-01-01", periods=n_days * 24, freq="h", tz="UTC")
    n = len(idx)

    # Base price level
    base = 65.0

    # Annual seasonality: higher in winter, lower in spring/autumn
    day_of_year = np.array(idx.dayofyear)
    annual = 20 * np.cos(2 * np.pi * (day_of_year - 355) / 365)  # peak ~Dec 21

    # Daily seasonality: double peak (morning + evening)
    hour = np.array(idx.hour)
    morning_peak = 15 * np.exp(-0.5 * ((hour - 9) / 2) ** 2)
    evening_peak = 18 * np.exp(-0.5 * ((hour - 20) / 2) ** 2)
    overnight_trough = -12 * np.exp(-0.5 * ((hour - 4) / 3) ** 2)
    daily = morning_peak + evening_peak + overnight_trough

    # Weekly seasonality: lower on weekends
    dow = np.array(idx.dayofweek)
    weekly = np.where(dow >= 5, -10.0, 0.0)  # -10 EUR on Sat/Sun

    # Slow commodity drift + one shock event
    trend = 0.003 * np.arange(n)  # gradual upward drift
    shock_start = n // 3
    shock = np.zeros(n)
    shock[shock_start:shock_start + 14 * 24] = 35.0  # 2-week price shock

    # Noise
    noise = rng.normal(0, 8, n)

    prices = base + annual + daily + weekly + trend + shock + noise
    prices = np.clip(prices, -20, 250)  # realistic range for Spanish market

    return pd.DataFrame({"day_ahead_price": prices}, index=idx)


ree_df = make_synthetic_prices(n_days=730)

print(f"Dataset: {len(ree_df):,} hourly samples")
print(f"Period:  {ree_df.index[0]} → {ree_df.index[-1]}")
print(f"\nPrice statistics:")
print(ree_df["day_ahead_price"].describe().round(2))

In [ ]:
# Quick visual check
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# Full series
ree_df["day_ahead_price"].plot(ax=axes[0], linewidth=0.5, color="steelblue")
axes[0].set_title("Synthetic electricity prices — full series (2 years)")
axes[0].set_ylabel("EUR/MWh")

# One week zoom
ree_df["day_ahead_price"].iloc[:168].plot(ax=axes[1], linewidth=1.2, color="steelblue")
axes[1].set_title("First week — daily seasonality visible")
axes[1].set_ylabel("EUR/MWh")

plt.tight_layout()
plt.show()

## Step 3 — Build features

`build_features()` transforms the raw price series into 100+ ML-ready features:
- Cyclical time encodings (hour, day-of-week, month)
- Price lags (1h, 2h, 3h, 24h, 48h, 72h, 168h)
- Rolling statistics (7d, 30d mean/std)
- Spanish holiday flags
- (Optional) Weather interaction features — skipped here since no weather data provided

In [ ]:
from src.data.feature_engineering import build_features

features_df = build_features(
    ree_df,
    weather_df=None,          # optional: daily weather DataFrame
    commodity_df=None,        # optional: daily gas/oil/carbon prices
    weather_hourly_df=None,   # optional: hourly Open-Meteo data (recommended)
)

feature_cols = [c for c in features_df.columns if c != "day_ahead_price"]
print(f"Features built: {len(feature_cols)} columns, {len(features_df):,} samples after lag warmup")
print(f"\nSample feature names: {feature_cols[:10]}")

## Step 4 — Train multi-horizon models

Each horizon group (`DA1`, `DA2`) gets its own model trained with walk-forward cross-validation.

- **DA1:** D+1 hours 00:00–11:00 (14–25h ahead from 10:00 UTC origin)
- **DA2:** D+1 hours 12:00–23:00 (26–37h ahead)

The trainer internally calls `build_direct_features()` to construct the lagged prediction dataset.

In [ ]:
from src.models.direct_trainer import DirectMultiHorizonTrainer

trainer = DirectMultiHorizonTrainer(
    model_type="histgb",   # "histgb" | "lgbm" | "xgboost"
    quantile=0.55,          # slightly above median — corrects right-skew bias
    feature_selection=False # set True to enable 2-stage feature pruning (slower)
)

print("Training DA1 + DA2 horizon groups...")
metrics = trainer.train_all(
    ree_df,
    n_splits=3,              # walk-forward CV folds
    run_mode="dayahead",     # "dayahead" | "strategic" | None (legacy)
    resolution="hourly",
)

print("\nTraining complete.")

## Step 5 — Inspect cross-validation metrics

In [ ]:
for group, fold_metrics in metrics.items():
    maes = [f["mae"] for f in fold_metrics if "mae" in f]
    if maes:
        print(f"  {group}: mean MAE = {np.mean(maes):.2f} EUR/MWh  "
              f"(folds: {', '.join(f'{m:.2f}' for m in maes)})")

In [ ]:
# Plot actual vs predicted on the last CV fold (DA1)
if "DA1" in trainer.metrics:
    last_fold = trainer.metrics["DA1"][-1]
    if "y_true" in last_fold and "y_pred" in last_fold:
        y_true = np.array(last_fold["y_true"])[:200]
        y_pred = np.array(last_fold["y_pred"])[:200]

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(y_true, label="Actual", linewidth=1.2, color="green")
        ax.plot(y_pred, label="Forecast", linewidth=1.2, linestyle="--", color="steelblue")
        ax.set_title("DA1 — Last CV fold: Actual vs Forecast (first 200 samples)")
        ax.set_ylabel("EUR/MWh")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Fold metrics don't include raw predictions — check trainer config.")

## Step 6 — Save trained models

In [ ]:
import joblib

for group_name, model in trainer.models.items():
    path = MODELS_DIR / f"direct_{group_name}.joblib"
    joblib.dump(model, path)
    print(f"Saved: {path}")

# Also save feature names (needed for inference)
feat_names_path = MODELS_DIR / "feature_names.joblib"
joblib.dump(trainer.feature_names, feat_names_path)
print(f"Saved: {feat_names_path}")

## Step 7 — Generate a forecast (inference)

To generate a real forecast you need a `DirectPredictor`, which requires the full data pipeline (REE API, weather API, commodity feeds). The cell below shows the interface; in practice, `ree_df`, `weather_df`, and `commodity_df` come from the data collectors.

```python
from src.models.direct_predictor import DirectPredictor

predictor = DirectPredictor(run_mode="dayahead")

origin_dt = pd.Timestamp("2025-06-01 10:00:00", tz="UTC")
forecasts = predictor.predict(
    origin_dt=origin_dt,
    ree_df=ree_df,          # recent hourly price + grid data
    weather_df=weather_df,  # recent weather
    commodity_df=commodity_df,
)
# Returns: list of {target_dt, predicted_price, lower_bound, upper_bound}
```

For a self-contained inference demo using only the features we already have:

In [ ]:
from src.models.direct_trainer import build_direct_features, HORIZON_GROUPS_DAYAHEAD

# Use the last 30 days as a held-out test window
test_df = ree_df.iloc[-30 * 24:]

# Build DA1 prediction samples for the test window
test_data = build_direct_features(
    ree_df,                         # full history (needed for lag warmup)
    horizons=HORIZON_GROUPS_DAYAHEAD["DA1"],
    weather_df=None,
    commodity_df=None,
    run_mode="dayahead",
)

if "DA1" in trainer.models and len(test_data) > 0:
    model = trainer.models["DA1"]
    feature_names = trainer.feature_names.get("DA1", [])

    # Keep only columns the model was trained on
    available = [c for c in feature_names if c in test_data.columns]
    X_test = test_data[available]
    y_pred = model.predict(X_test)
    y_true = test_data["target_price"].values

    mae = np.mean(np.abs(y_true - y_pred))
    print(f"Hold-out DA1 MAE (last 30 days): {mae:.2f} EUR/MWh")
    print(f"Samples: {len(y_pred)}")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(y_true[:168], label="Actual", linewidth=1.2, color="green")
    ax.plot(y_pred[:168], label="Forecast", linewidth=1.2, linestyle="--", color="steelblue")
    ax.set_title(f"DA1 hold-out forecast — first 7 days (MAE: {mae:.2f} EUR/MWh)")
    ax.set_ylabel("EUR/MWh")
    ax.legend()
    plt.tight_layout()
    plt.show()

---

## Next Steps

- **Add real data:** Replace synthetic `ree_df` with data from [REE ESIOS API](https://www.esios.ree.es/en/api) (public, free)
- **Add weather features:** Pass an hourly Open-Meteo DataFrame as `weather_hourly_df` — this typically reduces MAE by ~15%
- **Add commodity features:** TTF gas, Brent crude, EU ETS carbon prices from Yahoo Finance / ICE
- **Train strategic models:** Set `run_mode="strategic"` for D+2 through D+7 predictions
- **Enable LSTM embeddings:** See `src/models/lstm_embedder.py` — pre-train the encoder, then pass `use_lstm=True` to the trainer
- **Enable feature selection:** Set `feature_selection=True` in `DirectMultiHorizonTrainer` (slower training, can improve generalization)

Full documentation: [epforecast.vercel.app](https://epforecast.vercel.app)